# Sampler diagnostics

Erik, 2026-11-18

MCMC draws represent the posterior distribution only when the chains have
converged and mixed, and a broken fit returns numbers that look no different
at first glance from a valid one. Catching sampling failures requires two
kinds of diagnostics. Parameter-level summaries measure convergence and
simulation precision from each parameter's draws through R-hat, effective
sample size (ESS), and Monte Carlo standard error (MCSE). Iteration-level HMC
records track how the sampler moved through the posterior by checking
divergent transitions, trajectories that hit the maximum tree depth, and the
energy diagnostic E-BFMI.

The examples below run these checks on an HSLS:09 regression fit that passes,
examine two fits built to fail, and then apply the same workflow to a
hierarchical model with one intercept per school context before plotting its
group estimates in caterpillar plots.

In [ ]:
import framework as fw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

fw.scaffold()
import bayes  # functions/bayes.py, on the path after scaffold()

sns.set_theme(style="whitegrid", palette="colorblind")

clean = fw.data_read("inputs.final.hsls_clean", dtype={"student_id": str})
hsls = bayes.analysis_sample(clean)
regression = bayes.model("regression")
grouped = bayes.model("regression_groups")

# One generator for the subsamples
rng = np.random.default_rng(606)

## Convergence and precision

For each model parameter, `fit.summary()` returns a row of posterior estimates
alongside three diagnostics of convergence and precision:

- **R-hat.** Stan splits each chain in half, replaces the draws with standard
  normal scores based on their pooled ranks across chains, and compares spread
  within chains to spread across chains (Vehtari et al., 2021). It also
  computes the same statistic on folded draws (absolute distances from the
  posterior median) and reports the larger of the two values, which catches
  chains that share a center while differing in spread. Chains sampling a
  common distribution produce an R-hat near 1. Any value above 1.01 indicates
  disagreement across chains, meaning the draws should not be used for
  inference.
- **ESS.** Because successive MCMC draws are correlated, they carry less
  information than an equal number of independent draws. ESS estimates how
  many independent draws would yield the same precision as the autocorrelated
  chains in hand. Bulk ESS measures precision for central summaries such as
  the mean or median, whereas tail ESS measures precision at outer quantiles
  (the 5th and 95th percentiles). Both should reach at least 100 per chain, or
  400 across four chains.
- **MCSE.** A finite chain estimates the posterior mean with some simulation
  noise, measured by MCSE at roughly the posterior standard deviation (SD)
  divided by the square root of ESS. For MCSE, Stan computes ESS directly on
  the raw draws instead of the rank-normalized values used for bulk ESS, so
  the two ESS figures can differ when sampling fails. MCSE should remain small
  relative to the posterior SD.

The next cell fits the baseline regression to HSLS:09 and displays this
summary table across all model parameters.

In [ ]:
fit = bayes.sample(regression, bayes.stan_data(hsls, bayes.WEAK))
columns = ["Mean", "MCSE", "StdDev", "5%", "50%", "95%", "ESS_bulk", "ESS_tail", "R_hat"]
fit.summary().loc[["alpha", "beta[1]", "beta[2]", "sigma"], columns].round(3)

## Sampler behavior

Alongside parameter draws, Stan logs iteration-level records from the HMC
trajectory simulator:

- **Divergent transitions.** When a simulated trajectory enters a region where
  the posterior curves too sharply for the discrete step size, numerical
  simulation of the particle's path breaks down. Because the sampler misses or
  misweights that region, even a few divergences can bias posterior estimates.
- **Maximum tree depth.** Stan's No-U-Turn Sampler (NUTS), an adaptive form of
  HMC, repeatedly doubles a trajectory's length until the path turns back on
  itself, stopping at 10 doublings by default. Reaching that cap lowers
  sampling efficiency without invalidating the draws, signaling only that
  crossing the posterior required longer trajectories than the cap allowed.
- **E-BFMI.** The energy Bayesian fraction of missing information compares
  step-to-step changes in total HMC energy against overall energy variation
  across the chain. Values below 0.3 warn that the sampler moves poorly across
  energy levels, often because of heavy tails or difficult posterior geometry.

The helper function below extracts these three diagnostics from Stan's sampler
columns and reports them for the baseline fit.

In [ ]:
def sampler_checks(fit, max_depth=10):
    draws = fit.draws_pd()
    rows = {}
    for chain, chain_draws in draws.groupby("chain__"):
        energy = chain_draws["energy__"].to_numpy()
        rows[f"Chain {chain:.0f}"] = {
            "Divergent transitions": int(chain_draws["divergent__"].sum()),
            "At maximum tree depth": int((chain_draws["treedepth__"] >= max_depth).sum()),
            "E-BFMI": np.sum(np.diff(energy) ** 2) / np.sum((energy - energy.mean()) ** 2),
            "Mean steps per iteration": chain_draws["n_leapfrog__"].mean(),
        }
    return pd.DataFrame(rows).T.round(2)


def sampler_totals(fit, max_depth=10):
    # One row for all chains: counts summed over chains, steps averaged over draws
    draws = fit.draws_pd()
    return pd.Series(
        {
            "Divergent transitions": int(draws["divergent__"].sum()),
            "At maximum tree depth": int((draws["treedepth__"] >= max_depth).sum()),
            "Mean steps per iteration": round(draws["n_leapfrog__"].mean(), 1),
        }
    )


sampler_checks(fit)

In [ ]:
# CmdStan's own check reports the same quantities in words
print(fit.diagnose())

## Trace and rank plots

A trace plot graphs each chain's parameter draws in iteration order.
Well-mixed chains overlap tightly and fluctuate like random noise around a
constant horizontal band, without trends or stuck stretches. A rank plot pools
draws across all chains, replaces the raw values with their overall ranks from
smallest to largest, and plots a separate histogram of ranks for each chain.
When every chain samples the same target distribution, ranks spread evenly
across chains and each histogram stays roughly flat. Dense trace plots can
hide subtle differences beneath overlapping lines, whereas uneven rank
histograms reveal them clearly (Vehtari et al., 2021). The figure below shows
trace plots and rank histograms for all four parameters of the baseline fit.

In [ ]:
def trace_and_rank(fit, names, title):
    draws = fit.draws_pd()
    fig, axes = plt.subplots(2, len(names), figsize=(3.4 * len(names), 5.5))
    # One set of bin edges for every chain, so the histograms are comparable
    edges = np.linspace(0.5, len(draws) + 0.5, 21)
    for j, name in enumerate(names):
        ranks = stats.rankdata(draws[name])
        for chain, chain_draws in draws.groupby("chain__"):
            axes[0, j].plot(chain_draws["iter__"].to_numpy(), chain_draws[name].to_numpy(), linewidth=0.4, alpha=0.8)
            axes[1, j].hist(
                ranks[draws["chain__"] == chain], bins=edges, histtype="step", linewidth=1.2, label=f"Chain {chain:.0f}"
            )
        axes[0, j].set_title(name)
        axes[0, j].set_xlabel("Iteration")
        axes[1, j].set_xlabel("Rank of draw, all chains pooled")
    axes[0, 0].set_ylabel("Draw")
    axes[1, 0].set_ylabel("Draws per bin")
    axes[1, -1].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()


trace_and_rank(fit, ["alpha", "beta[1]", "beta[2]", "sigma"], "Trace plots (top) and rank plots (bottom), HSLS:09 regression")

All four parameters pass every diagnostic check. R-hat rounds to 1.00, bulk
and tail ESS sit in the thousands, MCSE is a tiny fraction of each parameter's
posterior SD, and no transitions diverge. In the plots above, the four chains
overlap around stable levels in the traces and produce flat rank histograms.

## Failure 1: chains too short for an awkward posterior

To show how the diagnostics respond to poor convergence, the next fit changes
the baseline specification in three ways. It leaves base-year math uncentered,
as the T-score divided by 10. That places the intercept at a base-year T-score
of 0, five SD below the mean and far outside the observed student data, and creates strong posterior correlation between the intercept and the math slope.
It also replaces the weakly informative priors with flat priors, since the
baseline intercept prior assumed a centered predictor. Finally, it runs only 50
warmup and 50 sampling iterations per chain, giving HMC too little time to adapt
to that correlated geometry.

In [ ]:
# Base-year math still in SD units (the T-score over 10), but not centered at 50
uncentered = hsls.assign(w1_math_z=hsls["w1_math"] / 10)
short = bayes.sample(regression, bayes.stan_data(uncentered), iter_warmup=50, iter_sampling=50)
short.summary().loc[["alpha", "beta[1]", "beta[2]", "sigma"], columns].round(3)

In [ ]:
trace_and_rank(short, ["alpha", "beta[1]", "beta[2]", "sigma"], "Too few iterations and an uncentered predictor")

In [ ]:
# The posterior correlation between the intercept and the base-year math slope,
# with and without centering, from the converged fit. Without centering, the
# intercept is the expected score at a base-year score of 0, 5 SD below 50:
# alpha - 5 * beta[1].
good = bayes.draws(fit)
pd.Series(
    {
        "Centered: alpha and beta[1]": good["alpha"].corr(good["beta[1]"]),
        "Uncentered: alpha - 5 beta[1] and beta[1]": (good["alpha"] - 5 * good["beta[1]"]).corr(good["beta[1]"]),
    },
    name="Posterior correlation",
).round(3).to_frame()

The summary table flags the breakdown across every parameter, with R-hat above
1.01 and bulk ESS well below 400. In the accompanying plots, the traces fail
to settle onto a shared level and the rank histograms tilt unevenly across
chains.

The second table shows why leaving base-year math uncentered makes sampling
hard, drawing on the converged fit from earlier so sampling failure in the
short chains does not distort the comparison. Shifting the intercept to a
base-year math score of 0 leaves it almost perfectly correlated with the
slope, since any steeper regression line must start lower to pass through the
observed scores far above 0. Exploring the resulting narrow ridge takes long
chains and parameter scaling properly tuned during warmup. Reversing both
changes resolves the problem. Centering the predictor removes most of the
posterior correlation, and a full warmup gives the sampler room to adapt.
Running many more iterations on the uncentered model would also converge
eventually, though slowly.

## Failure 2: a funnel in a hierarchical model

The hierarchical model in `stan/regression_groups.stan` adds a separate
intercept for each school context, covering all 31 combinations of census
region, locale, and public or private school control observed in HSLS:09.
Instead of estimating each group in isolation, the model draws the group
intercepts $a_j$ from a shared Normal(0, $\tau$) prior with $\tau$ estimated
from the data, pulling sparse groups toward 0.

When each group has only a few students, this model produces a funnel-shaped
posterior. At large values of $\tau$, the group intercepts $a_j$ can spread
widely across the mouth of the funnel, while at small values of $\tau$ they
pinch tightly around 0 in its neck. Because HMC uses a single step size after
warmup, a step size tuned for the wide mouth is too coarse for the narrow
neck, causing chains to avoid the neck, get stuck inside it for long
stretches, or diverge when entering it. The Stan program implements two
mathematically equivalent parameterizations of this model. The centered form
samples each $a_j$ directly. The non-centered form instead samples
standardized offsets $z_j$ whose prior does not depend on $\tau$ and sets
$a_j = \tau z_j$, removing the funnel from the parameter space HMC explores.

The code below fits both parameterizations to a random subsample of 3 students
per school context and compares their convergence summaries and divergence
counts.

In [ ]:
REGION = {1: "Northeast", 2: "Midwest", 3: "South", 4: "West"}
LOCALE = {1: "City", 2: "Suburb", 3: "Town", 4: "Rural"}
CONTROL = {1: "Public", 0: "Private"}

hsls["context"] = (
    hsls["school_w1_region"].map(REGION)
    + ", "
    + hsls["school_w1_locale"].map(LOCALE)
    + ", "
    + hsls["school_w1_public"].map(CONTROL)
)
contexts = sorted(hsls["context"].unique())
hsls["group"] = hsls["context"].map({name: j + 1 for j, name in enumerate(contexts)})


def group_data(data, centered):
    return {
        **{key: value for key, value in bayes.stan_data(data, bayes.WEAK).items() if key not in ("prior", "prior_only", "predict")},
        "J": len(contexts),
        "group": data["group"].to_numpy(),
        "tau_rate": 5.0,  # prior mean of tau 0.2 SD (2 T-score points)
        "centered": int(centered),
    }


# The fits on the subsample use seed 12, which shows both failures in one run
few = hsls.groupby("group", group_keys=False).sample(3, random_state=rng)
few_fits = {
    "Centered": bayes.sample(grouped, group_data(few, centered=True), seed=12),
    "Non-centered": bayes.sample(grouped, group_data(few, centered=False), seed=12),
}
pd.DataFrame({name: sampler_totals(fit_few) for name, fit_few in few_fits.items()}).T

In [ ]:
def form_summary(fit_form):
    summary = fit_form.summary()
    intercepts = summary.loc[summary.index.str.startswith("a[")]
    return pd.Series(
        {
            "tau: R-hat": summary.loc["tau", "R_hat"],
            "tau: bulk ESS": summary.loc["tau", "ESS_bulk"],
            "Group intercepts: largest R-hat": intercepts["R_hat"].max(),
            "Group intercepts: R-hat above 1.01": int((intercepts["R_hat"] > 1.01).sum()),
            "Group intercepts: smallest bulk ESS": intercepts["ESS_bulk"].min(),
            "Slopes: largest R-hat": summary.loc[["beta[1]", "beta[2]"], "R_hat"].max(),
            "Slopes: smallest bulk ESS": summary.loc[["beta[1]", "beta[2]"], "ESS_bulk"].min(),
        }
    )


pd.DataFrame({name: form_summary(fit_few) for name, fit_few in few_fits.items()}).round(3)

In [ ]:
# Each panel plots the parameter the sampler moves in: a[1] in the centered
# form, the standardized offset z[1] in the non-centered form
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for (name, fit_few), ax, sampled in zip(few_fits.items(), axes, ["a[1]", "z[1]"]):
    draws = fit_few.draws_pd()
    ok = draws["divergent__"] == 0
    ax.scatter(draws.loc[ok, sampled], np.log(draws.loc[ok, "tau"]), s=3, alpha=0.3, label="Draw")
    ax.scatter(draws.loc[~ok, sampled], np.log(draws.loc[~ok, "tau"]), s=12, color="red", label="Divergent transition")
    ax.set_title(f"{name}: {int((~ok).sum())} divergent transitions")
    ax.set_xlabel(f"Group 1, {sampled}")
axes[0].set_ylabel("log(tau)")
axes[1].legend(loc="lower right")
fig.suptitle("What the sampler sees, 3 students per group: a funnel (left) and its non-centered form (right)")
fig.tight_layout()

The centered parameterization fails most sharply on $\tau$, returning an R-hat
above 1.01 and a bulk ESS far below 400. Because the chains move slowly into
and out of the funnel's neck, they disagree on $\tau$ and carry that
disagreement into all 31 group intercepts, pushing every group intercept
slightly past an R-hat of 1.01 even while the regression slopes pass.
Inspecting only the focal slope coefficients would miss this failure, which is
why convergence checks must cover every parameter in the model. Under the
non-centered parameterization, R-hat for $\tau$ drops near 1 with thousands of
effective draws.

Divergent transitions are the second signature of a posterior funnel. With so
little data per group, divergences can appear under either parameterization,
and any divergence matters because it marks a region of the posterior left
unrepresented in the draws. The standard first remedy is raising
`adapt_delta`, the target proposal acceptance probability that governs
step-size tuning during warmup, so Stan settles on a smaller step size. The
next cell refits the non-centered model with `adapt_delta` raised from its
default of 0.8 to 0.99.

In [ ]:
careful = bayes.sample(grouped, group_data(few, centered=False), seed=12, adapt_delta=0.99)
pd.DataFrame(
    {
        "Non-centered, adapt_delta 0.8": sampler_totals(few_fits["Non-centered"]),
        "Non-centered, adapt_delta 0.99": sampler_totals(careful),
    }
).T

## The hierarchical model on all students

Across all 18,623 students, sample sizes range from 39 to 2,356 per school
context. With that much information in every group, the data pin down each
intercept directly and the posterior funnel vanishes. Informative group data
often favor the centered parameterization, because tightly constrained group
intercepts force $z_j$ and $\tau$ to trade off strongly in the non-centered
posterior. In the full-sample comparison below, both forms pass every
diagnostic check with comparable effective sample sizes, and the remaining
plots use the centered fit.

In [ ]:
full_fits = {
    "Centered": bayes.sample(grouped, group_data(hsls, centered=True)),
    "Non-centered": bayes.sample(grouped, group_data(hsls, centered=False)),
}
comparison = {}
for name, fit_full in full_fits.items():
    summary = fit_full.summary()
    comparison[name] = {
        "Divergent transitions": sampler_totals(fit_full)["Divergent transitions"],
        "Largest R-hat": summary["R_hat"].max(),
        "Smallest bulk ESS": summary["ESS_bulk"].min(),
        "Smallest tail ESS": summary["ESS_tail"].min(),
        "Parameter with the smallest bulk ESS": summary["ESS_bulk"].idxmin(),
    }
pd.DataFrame(comparison).T

## Diagnostics for every parameter

Once a model estimates dozens of group-level parameters, reading a summary
table row by row makes isolated problems easy to overlook. Plotting R-hat and
bulk ESS across every parameter of the centered fit puts all parameters in
view at once. In the plot below, every parameter should fall below the R-hat
threshold of 1.01 and above the bulk ESS threshold of 400.

In [ ]:
full = full_fits["Centered"]
summary = full.summary().drop(index="lp__")
summary = summary.loc[~summary.index.str.startswith("z[")]

fig, axes = plt.subplots(1, 2, figsize=(11, 8), sharey=True)
order = summary.sort_values("ESS_bulk").index
axes[0].scatter(summary.loc[order, "R_hat"], range(len(order)), s=12)
axes[0].axvline(1.01, color="red", linestyle="--")
axes[0].set_xlabel("R-hat")
axes[1].scatter(summary.loc[order, "ESS_bulk"], range(len(order)), s=12)
axes[1].axvline(400, color="red", linestyle="--")
axes[1].set_xscale("log")
axes[1].set_xlabel("Bulk ESS (log scale)")
axes[0].set_yticks(range(len(order)))
axes[0].set_yticklabels(order, fontsize=7)
fig.suptitle("R-hat and bulk ESS for every parameter, centered hierarchical model (dashed: thresholds)")
fig.tight_layout()

## Caterpillar plots

A caterpillar plot displays one posterior interval per group, ordered from
lowest to highest posterior mean so the intervals step across the figure like
the segments of a caterpillar. Here each interval represents a school context's
intercept, measuring how far expected follow-up scores in that context sit above
or below the overall intercept, in SD units, among students with the same
base-year math score and socioeconomic status (SES). The plot below arranges all
31 school contexts along this sorted scale, with thick bars for the central 50%
posterior intervals and thin lines for the central 90% intervals, showing both
how contexts rank relative to 0 and how precisely each offset is estimated.

In [ ]:
draws_full = full.draws_pd()
a = draws_full[[f"a[{j + 1}]" for j in range(len(contexts))]]
a.columns = contexts
sizes = hsls["context"].value_counts()

caterpillar = pd.DataFrame(
    {
        "mean": a.mean(),
        "q05": a.quantile(0.05),
        "q25": a.quantile(0.25),
        "q75": a.quantile(0.75),
        "q95": a.quantile(0.95),
        "n": sizes.reindex(contexts).to_numpy(),
    }
).sort_values("mean")

fig, ax = plt.subplots(figsize=(8, 9))
y = np.arange(len(caterpillar))
ax.hlines(y, caterpillar["q05"], caterpillar["q95"], color="C0", linewidth=1)
ax.hlines(y, caterpillar["q25"], caterpillar["q75"], color="C0", linewidth=4)
ax.plot(caterpillar["mean"], y, "o", color="black", markersize=3)
ax.axvline(0, color="grey", linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels([f"{name} (n = {n:,})" for name, n in zip(caterpillar.index, caterpillar["n"])], fontsize=8)
ax.set_xlabel("Group intercept: follow-up math SD units relative to the overall intercept")
ax.set_title("School-context intercepts, posterior mean with 50% and 90% intervals")
plt.tight_layout()

In [ ]:
# Partial pooling: a group's estimate is pulled toward 0 more when it has
# fewer students. The comparison is each group's mean residual around the
# hierarchical model's own common intercept and slopes (posterior means), the
# estimate a group would get with no pooling and the same reference point.
common = draws_full[["alpha", "beta[1]", "beta[2]"]].mean()
residual = hsls["w2_math_z"] - (
    common["alpha"] + common["beta[1]"] * hsls["w1_math_z"] + common["beta[2]"] * hsls["w1_ses"]
)
pooling = caterpillar.assign(raw=residual.groupby(hsls["context"]).mean().reindex(caterpillar.index))

fig, ax = plt.subplots(figsize=(6.5, 5))
points = ax.scatter(pooling["raw"], pooling["mean"], c=np.log10(pooling["n"]), cmap="viridis", s=30)
limits = [pooling[["raw", "mean"]].min().min() - 0.02, pooling[["raw", "mean"]].max().max() + 0.02]
ax.plot(limits, limits, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Mean residual in the group around the common line (no pooling)")
ax.set_ylabel("Posterior mean of the group intercept")
ax.set_title("Partial pooling pulls small groups toward 0")
fig.colorbar(points, label="Students in the group (log10)")
plt.tight_layout()
print(f"Posterior mean of tau, the SD of the group intercepts: {draws_full['tau'].mean():.3f} SD")

# How far each estimate moved toward 0, for the smallest and largest groups
pooling = pooling.assign(moved=pooling["raw"] - pooling["mean"]).sort_values("n")
pd.concat([pooling.head(5), pooling.tail(5)])[["n", "raw", "mean", "moved"]].round(3)

The comparison plot above contrasts those hierarchical intercepts with
no-pooling estimates computed from each group's students alone, without a
shared prior. School contexts on the dashed reference line keep their
no-pooling estimates, while departures from that line reflect estimates pulled
toward the overall intercept of 0. The companion table lists the five smallest
and five largest school contexts alongside the distance each estimate moved
toward 0. Large groups barely shift because their sample sizes dominate the
prior. Small groups with large raw means shift the most, pulled toward 0 by
the shared prior whose SD $\tau$ is estimated across all groups. That is
the same pull toward a prior center seen with small samples in
`02-ols-vs-bayes`, with the strength of the pull now learned from the data.